# Python to SQL, and back again
In this codealong will be shown how to create a relational database from pandas DataFrames.
> **To run this notebook you will need to work locally and not on colab.**

---
## Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [ ]:
# install if needed
#!pip install sqlalchemy
#!pip install pymysql

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.8 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 1.3 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 1.3 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.1 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.2 MB/s  0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ---------------------------------------- 0/3 [typing-extensions]
   ------------- -------------------------- 1/3 [g

In [30]:
import pandas as pd

## 1.&nbsp; Creating the cities table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.

In [31]:
cities = ["Berlin", "Hamburg", "Munich"]
cities_df = pd.DataFrame({"city": cities})
cities_df

,city
0,Berlin
1,Hamburg
2,Munich


To start our DataFrame index from 1 instead of the default 0, we can manually overwrite the index attribute after creating the object.

In [32]:
cities_df.index = range(1, len(cities_df) + 1)
cities_df

,city
1,Berlin
2,Hamburg
3,Munich


Create a function that allow to append the cities.

In [33]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse    # for decimal coordinates


def cities_dataframe(cities):

  city_data = []

  for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_latitude = city_soup.find(class_="latitude").get_text()
    city_longitude = city_soup.find(class_="longitude").get_text()
    country = city_soup.find(class_="infobox-data").get_text()

    # keep track of data per city
    city_data.append({"city": city,
                    "country": country,
                    "latitude": parse(city_latitude), # latitude in decimal format
                    "longitude": parse(city_longitude), # longitude in decimal format
                    })

  return pd.DataFrame(city_data)

In [34]:
list_of_cities = ["Berlin", "Hamburg", "Munich"]

cities_df = cities_dataframe(list_of_cities)
cities_df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


## 2.&nbsp; Creating the matching cities table with SQL 💻

-- Create the 'cities' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each city
    city VARCHAR(255) NOT NULL, -- Name of the city
    PRIMARY KEY (city_id) -- Primary key to uniquely identify each author
);

## 3.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

In [35]:
import os
from dotenv import load_dotenv
schema = "sql_gans"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [36]:
cities_df.to_sql('cities',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

Now, have a look at the table `cities` in MySQL Workbench, you should see that the names of the cities have appeared.

---
## 4.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [70]:
cities_from_sql = pd.read_sql("cities", con=connection_string)
cities_from_sql

,city_id,city,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [71]:
pd.read_sql("""
            SELECT DISTINCT city
            FROM cities
            """,
            con=connection_string)

,city
0,Berlin
1,Hamburg
2,Munich


---
## 5.&nbsp; Preparing and sending the populations table 
By extracting the cities table from our SQL database, we gain access to the unique identifier `city_id` assigned to each city. These `city_id`'s serve as pointers to their corresponding city records, allowing us to seamlessly link the `city_id`'s in the populations table to their respective cities in the cities table, thereby completing the populations table.

In [ ]:
# ensure you are getting BOTH the ID and the Name
cities_from_sql = pd.read_sql("SELECT city_id, city FROM cities", con=connection_string)

# DEBUG: Check if this is empty
print(f"Total cities found in SQL: {len(cities_from_sql)}")
print(cities_from_sql.head())


Total cities found in SQL: 3
   city_id     city
0        1   Berlin
1        2  Hamburg
2        3   Munich


In [73]:
cities = ["Berlin", "Hamburg", "Munich"]
population = [3850809, 1945532, 1512491]
year = [2024, 2024, 2024]
populations_df = pd.DataFrame({"city": cities,
                          "population": population,
                          "year_data_retrieved": year })
populations_df

,city,population,year_data_retrieved
0,Berlin,3850809,2024
1,Hamburg,1945532,2024
2,Munich,1512491,2024


In [74]:
# normalize names to ensure the merge actually finds matches
populations_df['city'] = populations_df['city'].str.strip().str.title()
cities_from_sql['city'] = cities_from_sql['city'].str.strip().str.title()

In [75]:
# merge to bring in 'city_id' based on a common column (like 'city_name')
populations_df = pd.merge(
    populations_df,
    cities_from_sql[['city_id', 'city']],
    on='city',
    how='inner'
)

In [76]:
populations_df

,city,population,year_data_retrieved,city_id
0,Berlin,3850809,2024,1
1,Hamburg,1945532,2024,2
2,Munich,1512491,2024,3


In [77]:
# final selection (only columns the SQL table 'populations' expects)
# do not use .drop() here, just select what you need
final_populations_to_sql = populations_df[['city_id', 'population', 'year_data_retrieved']]


In [78]:
schema = "sql_gans"
host = "127.0.0.1"
user = "root"
password = os.getenv("password")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [79]:
populations_df[['city_id', 'population', 'year_data_retrieved']].to_sql(
    name='populations',      # You must specify the SQL table name here
    con=connection_string,    # Your database connection
    if_exists='append',       # Add data to existing table
    index=False               # Do not write the DataFrame index as a column
)

3

In [80]:
print(populations_df.columns.tolist())

['city', 'population', 'year_data_retrieved', 'city_id']


---
## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [81]:
populations_from_sql = pd.read_sql("populations", con=connection_string)
populations_from_sql

,city_id,population,year_data_retrieved
0,1,3850809,2024
1,2,1945532,2024
2,3,1512491,2024


In [82]:
pd.read_sql("""
            SELECT city_id AS CITY_ID, FORMAT(population, 0) AS Population, year_data_retrieved AS Year_Data_Retrieved FROM populations
            """,
            con=connection_string)

,CITY_ID,Population,Year_Data_Retrieved
0,1,"3,850,809",2024
1,2,"1,945,532",2024
2,3,"1,512,491",2024
